# Precise GNSS Positioning with Differential Carrier Phase Measurements

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/DifferentialCarrierPhaseExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview
Carrier-phase precise GNSS positioning is a high-accuracy navigation technique that achieves centimeter-level precision by measuring the phase of the satellite's **radio carrier wave** rather than relying solely on the coarser "code" data used by standard GPS. Because the wavelength of the carrier signal is much shorter (roughly 19 cm for the L1 band) than the bits in the modulation code, the receiver can determine the distance to a satellite with far greater resolution. However, this method introduces the **integer ambiguity** problem—the challenge of determining exactly how many full wavelengths exist between the satellite and the antenna. To solve this, advanced algorithms and correction data from reference stations (such as in **Real-Time Kinematic** or **Precise Point Positioning**) are used to "fix" the ambiguity, transforming a standard meter-level position into a hyper-accurate measurement suitable for surveying, autonomous vehicles, and precision agriculture.

GNSS positioning problems can be encoded as factor graphs in GTSAM; this notebook extends the groundwork from [DifferentialPseudorangeExample.ipynb](https://borglab.github.io/gtsam/differentialpseudorangeexample/) to improve receiver positioning accuracy using both corrections from a nearby reference receiver and carrier-phase measurements 

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

In [ ]:
try:
    import google.colab
    %pip install --quiet pyrtklib numpy gtsam-develop pyproj tabulate folium
    !wget https://raw.githubusercontent.com/borglab/gtsam/refs/heads/develop/python/gtsam/examples/gnss_utils.py
except ImportError:
    pass

import folium
import gnss_utils
import gtsam
from gtsam.symbol_shorthand import B, C, X, N, D
import matplotlib.pyplot as plt
import numpy as np
from pyproj import Transformer
import pyrtklib as rtklib
from tabulate import tabulate

# Set up ecef -> lla converter:
ecef2lla = Transformer.from_crs("epsg:4978", "epsg:4326", always_xy=True)

np.set_printoptions(precision=4, suppress=True, linewidth=150)

## Premise
Our goal here is to precisely determine the position of the [P222](https://geodesy.noaa.gov/CORS/ncn_station_pages/index.html?stationID=p222) CORS station using differential code and carrier phase corrections from a nearby sister station [ZOA1](https://geodesy.noaa.gov/CORS/ncn_station_pages/index.html?stationID=ZOA1). Both P222 and ZOA1 positions are actually known a-priori, but for this exercise, we'll assume P222's position is unknown, and ZOA1 is a base station with known coordinates.

In [ ]:
print("=== ZOA1 Data ===")
print("Downloading and extracting...", end="")
zoa1_data = gnss_utils.loadCORSRINEX([
    "https://noaa-cors-pds.s3.amazonaws.com/rinex/2026/018/brdc0180.26n.gz",
    "https://noaa-cors-pds.s3.amazonaws.com/rinex/2026/018/zoa1/zoa10180.26o.gz"
])
print("done!")
print(zoa1_data)
print("done!\n")

print("=== P222 Data ===")
print("Downloading and extracting...", end="")
p222_data = gnss_utils.loadCORSRINEX([
    "https://noaa-cors-pds.s3.amazonaws.com/rinex/2026/018/brdc0180.26n.gz",
    "https://noaa-cors-pds.s3.amazonaws.com/rinex/2026/018/p222/p2220180.26o.gz"
])
print("done!")
print(p222_data)
print("done!\n")

## Initialization
Standard single-point and code-phase differential positioning techniques can be used to bootstrap an initial estimate of receiver position. For brevity, we'll borrow the results from [DifferentialPseudorangeExample.ipynb](https://borglab.github.io/gtsam/differentialpseudorangeexample/) and begin our exploration assuming all previous steps from the differential pseudorange example have been completed.

In [ ]:
p222_initial_pos = np.array([-2689640.94091633, -4290437.12345389,  3865051.53927859])

## Differential Positioning
Similar to the differential code-phase example, differential carrier-phase positioning assumes the base station has a known prior on its [location](https://noaa-cors-pds.s3.amazonaws.com/coord/coord_20/zoa1_20.coord.txt). The existing differential code-phase factor structure remains unchanged while additional carrier-phase factors are added to apply fine-grained corrections to the receiver's estimate. An additional complication with the integer-ambiguities come into play, however: Carrier-phase is measured in units of _cycles_, not meters, and the receiver does not know how many whole wavelength cycles span between itself and the satellite. A large part of the carrier-phase positioning problem is therefore occupied with the estimation of those whole wavelength cycles between the satellite and receiver.

### Integer Ambiguity Intuition


In [ ]:
zoa1_bias_key = B(1)
zoa1_pos_key = X(1)
zoa1_ref_pos = np.array([-2684436.824, -4293336.957, 3865351.638])  # ZOA1's surveyed reference position.
nm = gtsam.noiseModel.Diagonal.Sigmas(np.array([1.0]))
graph = gtsam.NonlinearFactorGraph()

initial_values = gtsam.Values()
initial_values.insert(zoa1_pos_key, zoa1_ref_pos)
initial_values.insert(zoa1_bias_key, 0.0)

Apply a prior factor constraint on zoa1's reference position. The noise model assumes the reference position is established to within 1 cm of precision.

In [ ]:
graph.add(
    gtsam.PriorFactorVector(
        zoa1_pos_key,
        zoa1_ref_pos,
        gtsam.noiseModel.Isotropic.Sigma(3, 0.0005)
    )
)

In [ ]:
# Prepare data structures for bookkeeping factor graph nodes:
sat_data = {} # dict map from epoch -> collection of sats
i = 0
for epoch, measurement_epoch in gnss_utils.iterateEpochs(zoa1_data, n=100):
    sat_data[epoch] = {}
    for j, obsd in enumerate(measurement_epoch.sat_obs):            
        # Apply factor:
        code_correction_key = C(i)
        carrier_correction_key = D(i)
        whole_cycles_key = N(i)
        i += 1
        initial_values.insert(code_correction_key, 0.0)
        initial_values.insert(carrier_correction_key, 0.0)
        initial_values.insert(whole_cycles_key, 0.0)
        if epoch == 0:
            graph.add(
                gtsam.PriorFactorDouble(code_correction_key, 0.0)
            )
            graph.add(
                gtsam.PriorFactorDouble(carrier_correction_key, 0.0)
            )
            graph.add(
                gtsam.PriorFactorDouble(whole_cycles_key, 0.0)
            )
        sat_data[epoch][obsd.sat] = (code_correction_key, carrier_correction_key, whole_cycles_key)
    
        # Code-phase differential factor:
        graph.add(
            gtsam.DifferentialPseudorangeFactor(
                zoa1_pos_key, zoa1_bias_key, code_correction_key,
                obsd.pseudorange, obsd.sat_pos, obsd.sat_bias, nm
            )
        )
    
        # Carrier-phase differential factor:
        graph.add(
            gtsam.DifferentialCarrierPhaseFactor(
                zoa1_pos_key, zoa1_bias_key, carrier_correction_key, whole_cycles_key,
                obsd.carrier_phase, obsd.sat_pos, obsd.sat_bias, nm
            )
        )

    # Connect the atmospheric correction and integer ambiguity nodes
    # with the previous epoch:
    if epoch > 0:
        for sat, node_keys in sat_data[epoch].items():
            if sat not in sat_data[epoch-1]:
                continue
                
            prev_code_correction_key, prev_carrier_correction_key, prev_whole_cycles_key = sat_data[epoch-1][sat]
            code_correction_key, carrier_correction_key, whole_cycles_key = node_keys
            graph.add(
                gtsam.BetweenFactorDouble(
                    prev_code_correction_key, code_correction_key, 0.0,
                    gtsam.noiseModel.Isotropic.Sigma(1, 0.01)
                )
            )
            graph.add(
                gtsam.BetweenFactorDouble(
                    prev_carrier_correction_key, carrier_correction_key, 0.0,
                    gtsam.noiseModel.Isotropic.Sigma(1, 0.01)
                )
            )
            graph.add(
                gtsam.BetweenFactorDouble(
                    prev_whole_cycles_key, whole_cycles_key, 0.0,
                    gtsam.noiseModel.Isotropic.Sigma(1, 0.1)
                )
            )

        # Iteratively update the estimates as observations are fused:
        if epoch % 100 == 0:    
            lm = gtsam.LevenbergMarquardtOptimizer(graph, initial_values)
            initial_values = lm.optimize()
    
    
# Solve the system:
lm = gtsam.LevenbergMarquardtOptimizer(graph, initial_values)
result = lm.optimize()

# Plot the integer ambiguities over epoch:
sat_amb = {}
for epoch, epoch_data in sat_data.items():
    for sat, node_keys in epoch_data.items():
        code_correction_key, carrier_correction_key, whole_cycles_key = node_keys
        pt = [epoch, result.atDouble(whole_cycles_key)]
        if sat in sat_amb:
            sat_amb[sat].append(pt)
        else:
            sat_amb[sat] = [pt]

for sat, amb_trace in sat_amb.items():
    amb_npy = np.array(amb_trace)
    plt.plot(amb_npy[:, 0], amb_npy[:, 1], label=str(sat))

plt.legend(loc='upper left')
plt.grid()
plt.xlabel("Epochs")
plt.ylabel("Phase Ambiguity")
plt.ylim(-1000, 1000)
plt.show()


# TODO: Calculate covariances of the integer variables and attempt rtklib LAMBDA:
marginals = gtsam.Marginals(graph, initial_values)
keys = [nk[2] for ignore, nk in sat_data[len(sat_data)-1].items()]
joint_marginal = marginals.jointMarginalCovariance(keys)
full_joint_cov = joint_marginal.fullMatrix()
print("=== full_joint_cov ===")
print(full_joint_cov)
print()

## Sources
- [PseudorangeFactor.h](https://github.com/borglab/gtsam/blob/develop/gtsam/navigation/PseudorangeFactor.h)
- [PseudorangeFactor.cpp](https://github.com/borglab/gtsam/blob/develop/gtsam/navigation/PseudorangeFactor.cpp)
- [DifferentialPseudorangeExample.ipynb](https://borglab.github.io/gtsam/differentialpseudorangeexample/)